In [26]:
import torch
import matplotlib.pyplot as plt
import sys
from neuralop.models import UNO
from neuralop import Trainer
from neuralop.training import AdamW
from neuralop.data.datasets import load_darcy_flow_small
from neuralop.utils import count_model_params
from neuralop import LpLoss, H1Loss
from torch.utils.data import DataLoader, TensorDataset
import torch.optim as optim
from neuralop.models import FNO
import h5py
import numpy as np
from sklearn.model_selection import train_test_split

device = 'cpu'

In [27]:
datastore = h5py.File("C:/Users/arnab/OneDrive/Desktop/Study material/summer 24/Research/New equations/datasets/trials_many_alldata_new_eqution_halton.h5", 'r')
data_I = np.array(np.array(datastore["I"]))

data_c = np.array(np.array(datastore["c"]))
data_h = np.array(np.array(datastore["h"]))
t = np.linspace(0, 1, 601).reshape(601, 1)

In [28]:
train_label, test_label = train_test_split(range(len(data_I)), test_size=0.25, random_state=42)

data_i_train = data_I[train_label,:]
data_i_test = data_I[test_label,:]

data_c_train = data_c[train_label,:]
data_c_test = data_c[test_label,:]

data_h_train = data_h[train_label,:]
data_h_test = data_h[test_label,:]

In [29]:
data_i_train = torch.tensor(data_i_train, dtype=torch.float32)
data_i_test = torch.tensor(data_i_test, dtype=torch.float32)

data_c_train = torch.tensor(data_c_train, dtype=torch.float32)
data_c_test = torch.tensor(data_c_test, dtype=torch.float32)

data_h_train = torch.tensor(data_h_train, dtype=torch.float32)
data_h_test = torch.tensor(data_h_test, dtype=torch.float32)

In [30]:
# Define the FNO model
operator_c = FNO(
    n_modes=(16,),         
    hidden_channels=64,    
    in_channels=1,        
    out_channels=1         
)


X_train_fno = data_i_train.unsqueeze(1) 
Y_train_fno = data_c_train.unsqueeze(1) 
X_test_fno = data_i_test.unsqueeze(1)   
Y_test_fno = data_c_test.unsqueeze(1)   

# Define loss function and optimizer
criterion = torch.nn.MSELoss()
optimizer = optim.Adam(operator_c.parameters(), lr=0.001)

In [31]:
# Training loop
num_epochs = 15
batch_size = 256

# Create DataLoaders for batching
train_loader = torch.utils.data.DataLoader(
    list(zip(X_train_fno, Y_train_fno)), batch_size=batch_size, shuffle=True
)
test_loader = torch.utils.data.DataLoader(
    list(zip(X_test_fno, Y_test_fno)), batch_size=batch_size, shuffle=False
)

for epoch in range(num_epochs):
    operator_c.train()  
    epoch_loss = 0.0

    for inputs, targets in train_loader:
        optimizer.zero_grad() 
        predictions = operator_c(inputs)  
        loss = criterion(predictions, targets)  
        loss.backward()  
        optimizer.step() 
        epoch_loss += loss.item()
    
    print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {epoch_loss / len(train_loader)}")

# Evaluate on test set
operator_c.eval()
test_loss = 0.0
with torch.no_grad():
    for inputs, targets in test_loader:
        predictions = operator_c(inputs)
        loss = criterion(predictions, targets)
        test_loss += loss.item()
print(f"Test Loss: {test_loss / len(test_loader)}")

Epoch 1/15, Loss: 5.784298118752746
Epoch 2/15, Loss: 1.4572642322244316
Epoch 3/15, Loss: 0.8092000500348668
Epoch 4/15, Loss: 0.5572303888098947
Epoch 5/15, Loss: 0.43463428355870204
Epoch 6/15, Loss: 0.3679023681202652
Epoch 7/15, Loss: 0.32492985457275353
Epoch 8/15, Loss: 0.29036800472246815
Epoch 9/15, Loss: 0.24699648261817644
Epoch 10/15, Loss: 0.23202351896460152
Epoch 11/15, Loss: 0.2084222772278382
Epoch 12/15, Loss: 0.1861851399408239
Epoch 13/15, Loss: 0.1744131473911967
Epoch 14/15, Loss: 0.13358390121063843
Epoch 15/15, Loss: 0.15916207212032196
Test Loss: 0.13952272531585158


In [32]:
model_c.eval()
test_loss = 0.0
with torch.no_grad():
    for inputs, targets in test_dataloader_c:
        predictions = model_c(inputs)
        loss = criterion(predictions, targets)
        test_loss += loss.item()
print(f"Test Loss: {test_loss / len(test_dataloader_c)}")

NameError: name 'model_c' is not defined

In [ ]:
train_dataset_h = TensorDataset(data_i_train, data_h_train)
train_dataloader_h = DataLoader(train_dataset_h, batch_size=1024, shuffle=True)

test_dataset_h = TensorDataset(data_i_test, data_h_test)
test_dataloader_h = DataLoader(test_dataset_h, batch_size=1024, shuffle=True)

In [ ]:
model_h = FNO(
    n_modes=(16,),         
    hidden_channels=40,    
    in_channels=1,        
    out_channels=1         
)
model_h = model_c.to(device)

n_params = count_model_params(model_h)
print(f'\nOur model has {n_params} parameters.')
sys.stdout.flush()



Our model has 345665 parameters.


In [ ]:
num_epochs = 5
# batch_size = 32

for epoch in range(num_epochs):
    model_h.train()  
    epoch_loss = 0.0

    for inputs, targets in train_dataloader_h:
        optimizer.zero_grad() 
        predictions = model_h(inputs)  
        loss = criterion(predictions, targets)  
        loss.backward()  
        optimizer.step() 
        epoch_loss += loss.item()
    
    print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {epoch_loss / len(train_dataloader_h)}")

Epoch 1/5, Loss: 0.31314539268794583
Epoch 2/5, Loss: 0.06267611069105021
Epoch 3/5, Loss: 0.061953247184070144
Epoch 4/5, Loss: 0.061158889086871615
Epoch 5/5, Loss: 0.06013041021438634


In [ ]:
model_h.eval()
test_loss = 0.0
with torch.no_grad():
    for inputs, targets in test_dataloader_h:
        predictions = model_h(inputs)
        loss = criterion(predictions, targets)
        test_loss += loss.item()
print(f"Test Loss: {test_loss / len(test_dataloader_h)}")

Test Loss: 0.06038714985230139
